In [1]:
import subprocess
import time

import torch


def run_nvidia_smi():
    try:
        result = subprocess.run(
            ["nvidia-smi"],
            check=False,
            capture_output=True,
            text=True,
            timeout=10,
        )
    except FileNotFoundError:
        return False, "nvidia-smi was not found. Install the NVIDIA driver utilities."
    except subprocess.TimeoutExpired:
        return False, "nvidia-smi timed out. The NVIDIA driver may be stuck."

    output = (result.stdout + result.stderr).strip()
    return result.returncode == 0, output


print(f"PyTorch: {torch.__version__}")
print(f"PyTorch CUDA build: {torch.version.cuda}")

smi_ok, smi_output = run_nvidia_smi()
print(f"nvidia-smi works: {smi_ok}")
print(smi_output.splitlines()[0] if smi_output else "No nvidia-smi output")

cuda_available = torch.cuda.is_available()
device_count = torch.cuda.device_count() if cuda_available else 0
print(f"torch.cuda.is_available(): {cuda_available}")
print(f"CUDA device count: {device_count}")

if cuda_available:
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available to PyTorch right now; using CPU fallback.")
    if torch.version.cuda is None:
        print("This PyTorch install is CPU-only. Install a CUDA-enabled PyTorch build.")
    elif not smi_ok:
        print("Your PyTorch build has CUDA, but the NVIDIA driver is not reachable.")
        print(
            "Fix the system driver first: `nvidia-smi` must work before PyTorch can use the GPU."
        )


# Tensor size. Increase this if timings are too small/noisy on your machine.
N = 10_000_000


def benchmark(device):
    x = torch.rand(N, device=device)

    if device.type == "cuda":
        # Warm up kernels and make GPU timing honest.
        _ = torch.pow(10, torch.log10(x))
        torch.cuda.synchronize()

    start = time.perf_counter()
    y = torch.pow(10, torch.log10(x))

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    return elapsed, y


cpu_time, _ = benchmark(torch.device("cpu"))
print(f"CPU time: {cpu_time:.6f} sec")

if cuda_available:
    gpu_time, _ = benchmark(device)
    print(f"GPU time: {gpu_time:.6f} sec")
    print(f"Speedup: {cpu_time / gpu_time:.2f}x")


PyTorch: 2.11.0+cu130
PyTorch CUDA build: 13.0
nvidia-smi works: True
Mon May 11 15:29:39 2026       
torch.cuda.is_available(): True
CUDA device count: 1
Using GPU: NVIDIA GeForce RTX 4060
CPU time: 0.022388 sec
GPU time: 0.000688 sec
Speedup: 32.55x


In [ ]:
import time

import torch


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Run `nvidia-smi` and the diagnostic cell above first."
    )


MATRIX_SIZE = 4096
torch.manual_seed(42)


# ---------------- CPU matmul ----------------
a_cpu = torch.randn(MATRIX_SIZE, MATRIX_SIZE)
b_cpu = torch.randn(MATRIX_SIZE, MATRIX_SIZE)

start = time.perf_counter()
c_cpu = a_cpu @ b_cpu
cpu_time = time.perf_counter() - start

print(f"CPU matmul time: {cpu_time:.6f} sec")


# ---------------- GPU matmul ----------------
device = torch.device("cuda")
a_gpu = a_cpu.to(device)
b_gpu = b_cpu.to(device)

# Warm up the GPU kernel before timing.
_ = a_gpu @ b_gpu
torch.cuda.synchronize()

start = time.perf_counter()
c_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.perf_counter() - start

print(f"GPU matmul time: {gpu_time:.6f} sec")
print(f"Speedup: {cpu_time / gpu_time:.2f}x")
print(
    f"GPU result close to CPU: {torch.allclose(c_cpu, c_gpu.cpu(), atol=1e-2, rtol=1e-2)}"
)


CPU matmul time: 0.178438 sec
GPU matmul time: 0.016830 sec
Speedup: 10.60x
GPU result close to CPU: True


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(1000, 1000, device=device)
w = torch.randn(1000, 1000, device=device)

y = x @ w
